# WMT-Human — Preprocessing

This notebook handles data loading, cleaning, and preparation for the WMT-Human analysis.


In [ ]:
import pandas as pd
import numpy as np
import re


## Load Raw Data


In [ ]:
# Load raw data
df = pd.read_csv('../data/wmt-human_en_de_judged.csv')

print("=" * 50)
print("RAW DATA")
print("=" * 50)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()


## Data Cleaning


In [ ]:
# Drop unnecessary columns
columns_to_drop = [
    'Unnamed: 0', 'id', 'source', 'reference', 'translation',
    'mean_human_score', 'individual_human_scores', 'n_annotations',
    'GPT-MINI_as_a_judge', 'MIXTRAL_as_a_judge',
]

df = df.drop(columns=columns_to_drop)

print(f"Shape after dropping columns: {df.shape}")
print(f"Remaining columns: {list(df.columns)}")
df.head()


In [ ]:
# Check unique values before cleaning
print("Before cleaning:")
print(f"human #1 unique values: {df['human_#1'].unique()}")
print(f"human #2 unique values: {df['human_#2'].unique()}")
print(f"human #3 unique values: {df['human_#3'].unique()}")
print()
print("Mistral unique values: ", df['MISTRAL_as_a_judge'].unique())
print("LLAMA unique values: ", df['LLAMA_as_a_judge'].unique())
print("GPT-4o unique values: ", df['GPT_as_a_judge'].unique())


In [ ]:
def extract_valid_rating(series: pd.Series) -> pd.Series:
    """
    Extract leading number from strings in a Series,
    keeping only values between 0 and 6 inclusive.
    Non-matching or out-of-range values are returned as NaN.

    Parameters
    ----------
    series : pd.Series

    Returns
    -------
    pd.Series
        Series of integers in [0, 6] or NaN.
    """
    pattern = re.compile(r'^\s*(\d+)')

    values = []
    for item in series:
        text = str(item)
        match = pattern.match(text)
        if match:
            value = int(match.group(1))
            if 0 <= value <= 6:
                values.append(value)
            else:
                values.append(np.nan)
        else:
            values.append(np.nan)

    return pd.Series(values, index=series.index)

# Apply cleaning to all columns
for col in df.columns:
    df[col] = extract_valid_rating(df[col])

print("After cleaning:")
print("Mistral unique values: ", df['MISTRAL_as_a_judge'].unique())
print("LLAMA unique values: ", df['LLAMA_as_a_judge'].unique())
print("GPT-4o unique values: ", df['GPT_as_a_judge'].unique())


In [ ]:
# Cast to int where possible (keeps NaN as float)
df = df.astype(int, errors='ignore')

print(f"Final shape: {df.shape}")
print()
print("Final unique values:")
print("Mistral unique values: ", df['MISTRAL_as_a_judge'].unique())
print("LLAMA unique values: ", df['LLAMA_as_a_judge'].unique())
print("GPT-4o unique values: ", df['GPT_as_a_judge'].unique())
print()
print("Data types:")
print(df.dtypes)


## Save Cleaned Data


In [ ]:
# Save cleaned dataset
df.to_csv("../data/wmt-human_en_de_judged_cleaned.csv", index=False)
print(f"✅ Saved cleaned dataset to ../data/wmt-human_en_de_judged_cleaned.csv")
print(f"   Shape: {df.shape}")

print()
print("=" * 50)
print("PREPROCESSING COMPLETE")
print("=" * 50)
print()
print("Output file:")
print("  - ../data/wmt-human_en_de_judged_cleaned.csv")
